# Assignment 20 - Building a Recommendation System
**Name:** Shiza

**Dataset:** Custom movies dataset - 50 movies with title, genre, overview  
**Type:** Content-based recommendation using TF-IDF + Cosine Similarity

the streamlit app is in app.py

---
## Part 1 - Data Preprocessing
### Task 1 - Load & Understand Dataset

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('movies.csv')
print('shape:', df.shape)
print('columns:', df.columns.tolist())

shape: (50, 3)
columns: ['title', 'genre', 'overview']


In [3]:
df.head()

,title,genre,overview
0,The Dark Knight,Action Crime Drama,A criminal mastermind known as the Joker wreak...
1,Inception,Action Sci-Fi Thriller,A thief who enters the dreams of others to ste...
2,Interstellar,Adventure Drama Sci-Fi,A team of astronauts travel through a wormhole...
3,The Matrix,Action Sci-Fi,A computer hacker learns from mysterious rebel...
4,Avengers Endgame,Action Adventure Sci-Fi,After the devastating events the remaining Ave...


In [4]:
df.isnull().sum()

title       0
genre       0
overview    0
dtype: int64

In [5]:
# text column used for recommendations = overview + genre combined
print('text columns for recommendation: genre and overview')
print('\nsample overview:')
print(df['overview'][0])

text columns for recommendation: genre and overview

sample overview:
A criminal mastermind known as the Joker wreaks havoc on Gotham City and Batman must accept one of the greatest tests of his ability to fight injustice


---
### Task 2 - Text Preprocessing

In [6]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

# combining genre and overview into one text column
df['combined'] = df['genre'].fillna('') + ' ' + df['overview'].fillna('')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['combined'].apply(clean_text)
df[['title','clean_text']].head(4)

[nltk_data] Error loading stopwords: <urlopen error [Errno 61]
[nltk_data]     Connection refused>


,title,clean_text
0,The Dark Knight,action crime drama criminal mastermind known j...
1,Inception,action scifi thriller thief enters dreams othe...
2,Interstellar,adventure drama scifi team astronauts travel w...
3,The Matrix,action scifi computer hacker learns mysterious...


In [7]:
print('sample clean text:')
print(df['clean_text'][0])

sample clean text:
action crime drama criminal mastermind known joker wreaks havoc gotham city batman must accept one greatest tests ability fight injustice


---
## Part 2 - Text Vectorization
### Task 3 - TF-IDF Vectorization

In [8]:
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print('tfidf matrix shape:', tfidf_matrix.shape)

tfidf matrix shape: (50, 500)


In [9]:
print('vocab size:', len(tfidf.vocabulary_))
print('some features:', list(tfidf.vocabulary_.keys())[:10])

vocab size: 500
some features: ['action', 'crime', 'drama', 'mastermind', 'wreaks', 'must', 'one', 'ability', 'fight', 'action crime']


---
### Task 4 - Similarity Computation

In [10]:
sim_matrix = cosine_similarity(tfidf_matrix)
print('similarity matrix shape:', sim_matrix.shape)

similarity matrix shape: (50, 50)


In [11]:
# viewing similarity of first movie with others
sim_df = pd.DataFrame(sim_matrix[:5, :5],
                      index=df['title'][:5],
                      columns=df['title'][:5])
sim_df.round(2)

title,The Dark Knight,Inception,Interstellar,The Matrix,Avengers Endgame
title,,,,,
The Dark Knight,1.00,0.01,0.01,0.01,0.01
Inception,0.01,1.00,0.02,0.10,0.04
Interstellar,0.01,0.02,1.00,0.02,0.03
The Matrix,0.01,0.10,0.02,1.00,0.03
Avengers Endgame,0.01,0.04,0.03,0.03,1.00


cosine similarity measures the angle between two vectors. if two movies have similar words in their overview and genre, their vectors point in similar directions so cosine similarity is close to 1. it works well for text because it ignores vector length and only cares about direction.

---
## Part 3 - Recommendation Logic
### Task 5 - Build Recommendation Function

In [12]:
def recommend(item_name, top_n=5):
    # find index of selected movie
    matches = df[df['title'].str.lower() == item_name.lower()]
    if len(matches) == 0:
        print('movie not found')
        return []
    
    idx = matches.index[0]
    
    # get similarity scores for this movie
    scores = list(enumerate(sim_matrix[idx]))
    
    # sort by score descending
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    # skip the movie itself (index 0 after sorting = same movie)
    top = scores[1:top_n+1]
    
    # return top n movie titles
    result = [df['title'].iloc[i] for i, s in top]
    return result

In [13]:
# testing with 3 different movies
print('recommendations for The Dark Knight:')
print(recommend('The Dark Knight'))

recommendations for The Dark Knight:
['The Godfather', 'Fast and Furious', 'John Wick', 'Pulp Fiction', 'The Shawshank Redemption']


In [14]:
print('\nrecommendations for Inception:')
print(recommend('Inception'))


recommendations for Inception:
['The Matrix', 'Mission Impossible', 'John Wick', 'Iron Man', 'Memento']


In [15]:
print('\nrecommendations for Toy Story:')
print(recommend('Toy Story'))


recommendations for Toy Story:
['Finding Nemo', 'Frozen', 'The Lion King', 'Guardians of the Galaxy', 'Interstellar']


In [16]:
print('\nrecommendations for The Notebook:')
print(recommend('The Notebook'))


recommendations for The Notebook:
['Titanic', 'Casablanca', 'Forrest Gump', 'Good Will Hunting', 'Dead Poets Society']
